# 3D Toric-Code NQS — hyperparameter sweep on Colab

Runs the **same** `Three_TC.train` pipeline as the NERSC GPU job array. On Colab
you run **each config as its own cell** — one fresh `!python` process per cell,
which is the pattern that runs reliably here (the automated loop could skip runs).

**Run cells 1–5 top to bottom once** (setup), then use the **run cells** at the
bottom: copy one, change `NONINV`/`INV`/`KERNEL` and the training hyperparameters, and run it.

The default network is **`ToricCNN_gridinv`** — the 2D-paper architecture generalised
to 3D: edge spins → geometry-exact noninv blocks → per-channel **Wilson** 4-product
(flux, exactly `A_v`-invariant) → fold the flux field onto the cube-cell grid →
**standard `nn.Conv3D`** invariant block with kernel scaled to `L` → masked mean.
The grid invariant block is what lets the kernel cheaply span the system for the
topological long-range order. Each run does dense-QGT Stochastic Reconfiguration and
prints, every step, `E ± ΔE`, the spread `√Var`, and the figure of merit
**`delta = |E − E_exact| / |E_exact|`** (also logged to W&B). Runs land in an
arch-named folder `outputs/gridinv/…`.

> First: **Runtime → Change runtime type → GPU**, then run cell 1 to confirm.

**Boundary conditions.** Set `BC = "PBC"` or `"OBC"` in the configure cell. PBC uses the hardcoded `--hz_preset` ED reference; OBC (L=2, N=12) diagonalizes the exact ground state **on the fly** and passes it as `--exact_E0`, so `delta` works for both.

**System size `L` — and what changes past L=2.** Just set `L` in the configure cell;
the VMC/sampling/network stack scales with `L` (the `gridinv` kernel auto-grows to `L`;
the sampler `n_sweeps` auto-scales to `2N`). The **only** thing that does *not* scale is
the exact reference: exact diagonalization is tractable only at **L=2** (PBC N=24 / OBC
N=12). At **L≥3** the Hilbert space explodes (PBC N=81 → 2⁸¹, OBC N=54 → 2⁵⁴), so there
is **no `E_exact`** — the configure cell detects this, disables `delta` (it prints
`None`), and the comparison becomes the **final variational energy** (lower wins). The
L≥3 headline is the **energy gap between the symmetry-aware net and the symmetry-unaware
`GeoCNN`** at matched params, plus training stability (`R̂ ≈ 1`, shrinking `√Var`).

In [1]:
# 1. Confirm a GPU is attached (Runtime -> Change runtime type -> GPU).
!nvidia-smi -L || echo "NO GPU -- switch the runtime to GPU before continuing."

GPU 0: NVIDIA L4 (UUID: GPU-5b51bb2b-13a2-bb52-fd71-4ed14e1f8d43)


In [2]:
# 2. Clone (or update) the repo into the Colab VM. Idempotent: safe to re-run.
import os
REPO_URL = "https://github.com/SanzharBissenali/ThreeD_TC.git"
REPO_DIR = "/content/ThreeD_TC"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git pull --ff-only || true
print("cwd:", os.getcwd())

Cloning into '/content/ThreeD_TC'...
remote: Enumerating objects: 409, done.
remote: Counting objects: 100% (409/409), done.
remote: Compressing objects: 100% (247/247), done.
remote: Total 409 (delta 186), reused 374 (delta 151), pack-reused 0 (from 0)
Receiving objects: 100% (409/409), 1.06 MiB | 1.08 MiB/s, done.
Resolving deltas: 100% (186/186), done.
Already up to date.
cwd: /content/ThreeD_TC


In [3]:
# 3. Install the NQS stack, pinned to match the NERSC env (parity of results).
#    Colab may print dependency-conflict warnings for unrelated preinstalled
#    packages (tensorflow, etc.) -- those are harmless for this pipeline.
!pip install -q jax==0.5.2 jaxlib==0.5.1 \
    jax-cuda12-plugin==0.5.1 jax-cuda12-pjrt==0.5.1 \
    netket==3.16.1.post1 flax==0.10.4 optax numba wandb tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.2/105.2 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 695.1/695.1 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.8/451.8 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.3/354.3 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.0/101.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 142.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.3/529.3 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.4 MB/s eta 0:00:00


In [4]:
# 4. GATE: make sure JAX actually sees the GPU (expect platform 'gpu').
#    If this fails: re-run cell 3, then Runtime -> Restart session, then re-run
#    from cell 2. (x64 is already enabled inside Three_TC/train.py.)
import os
# Allocate GPU memory on demand instead of grabbing ~75% up front -- keeps
# back-to-back runs from fighting over VRAM. Set BEFORE importing jax.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
import jax
print("devices:", jax.devices(), "| backend:", jax.default_backend())
assert jax.default_backend() == "gpu", "JAX is not on the GPU -- see the note above."

devices: [CudaDevice(id=0)] | backend: gpu


## Weights & Biases

`WANDB = True` logs every step's `delta` / `energy` / `energy_abs_err` to a shared
group so all configs plot side by side (a parallel-coordinates plot over `delta` is
the intended view). Use a group name **distinct from your NERSC group** so the two
don't mix — same project, so they're still visible together.

Set `WANDB = False` to skip it entirely and just watch the per-step stdout.

In [5]:
# 5. W&B auth. Set WANDB = False to skip and rely on stdout only.
# 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

WANDB = True
if WANDB:
    import wandb
    wandb.login()   # paste your API key when prompted

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: ERROR Invalid API key: API key may only contain the letters A-Z, digits and underscores.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


...

wandb: WARNING Invalid choice


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sbissena (models-california-institute-of-technology-caltech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Configure — fixed args + a training menu  ← edit here

Set the shared args, the architecture, and the per-run output folder once; every run
cell below picks them up. `ARCH_FLAGS = ""` uses the **code default** net
(non-invariant `[4,4]`, invariant `[4,4,1]`, ~2.6k params) — set it to override. The
printed menu lists **training**-hyperparameter combos to copy into a run cell.

Pick `BC` (`PBC`/`OBC`) here as well: OBC computes its exact `E0` on the fly (cheap at L=2) and writes to a separate `colab-obc-…` folder, while PBC is unchanged.

In [ ]:
# === EDIT: fixed args shared by every run ==================================
BC        = "OBC"      # boundary conditions: "PBC" | "OBC"
L         = 2          # linear size. L=2 has an exact reference; L>=3 does not (ED intractable)
HX        = 0.2        # transverse field (keep 0.2 for the L=2 PBC presets)
HZ        = 0.2        # the hz value (used directly except for L=2 PBC, which takes HZ_PRESET)
HZ_PRESET = "hard"     # L=2 PBC ED reference point: hard | mid | easy  (ignored otherwise)
N_ITER    = 200
N_SAMPLES = 8192
N_CHAINS  = 512
N_SWEEPS  = 0          # 0 -> code default 2N (48 at L=2, 162 at L=3); set >0 to override
CHUNK     = 1024        # samples per chunk in the E_loc/grad step. REQUIRED at L>=3:
                        # the full N_SAMPLES batch x ~108 connected configs OOMs the
                        # GPU (~110 GiB). Lower if still OOM; 0 = no chunking (L=2 only).

# Architecture (default for this notebook): ToricCNN_gridinv — the 2D-paper net
# generalised to 3D. After the Wilson 4-product the flux field is folded onto the
# cube-cell grid (3 orientation channels) and run through a STANDARD nn.Conv3D
# invariant block whose kernel scales toward L (full span) for the topological
# long-range order. The pre-Wilson (noninv) block stays geometry-exact.
#   noninv [4,4]  (noninv_channels=4, n_noninv=2)
#   inv    [4,4]  grid-conv hidden widths (+ width-1 readout), kernel auto = L
# Override per run in the run cells below (NONINV / INV / KERNEL).
ARCH       = "ToricCNN_gridinv"
ARCH_TAG   = "gridinv"                 # label -> arch-named output folder + wandb group
ARCH_FLAGS = ""                        # "" = code defaults; run cells set these explicitly
# ===========================================================================

# Ground-truth / field spec. Exact diagonalization is only tractable at L=2
# (PBC N=24 / OBC N=12). At L>=3 the Hilbert space explodes (PBC N=81 -> 2^81,
# OBC N=54 -> 2^54), so there is NO exact reference: the `delta` FOM is disabled
# and runs are compared by their *final variational energy* instead (lower wins,
# by the variational principle). The headline at L>=3 is therefore the GAP between
# the symmetry-aware net and the symmetry-unaware GeoCNN at matched params, plus
# training stability (R_hat ~ 1, spread sqrt(Var) shrinking).
HAS_GROUND_TRUTH = (L == 2)

if not HAS_GROUND_TRUTH:
    # L>=3: no ED. Pass the field directly; no --exact_E0, so train.py reports
    # E +- dE and the spread but delta = None (compare archs by final E).
    FIELD_FLAGS = f"--hx {HX} --hz {HZ}"
    POINT_TAG   = f"L{L}_hx{HX}_hz{HZ}"
    print(f"L={L} {BC}: NO exact reference (ED intractable) -> compare archs by final E")
elif BC == "OBC":
    # L=2 OBC: diagonalize the N=12 Hamiltonian on the fly (2^12, a fraction of a
    # second) with the same verified ED as the repo, and pass E0 to --exact_E0.
    import numpy as np
    from scipy.sparse.linalg import eigsh
    from Three_TC.model.geometry import ThreeD_ToricCodeGeometry
    from model.exact_diag import hamiltonian_linop
    _geo = ThreeD_ToricCodeGeometry(L, L, L, bc="OBC")
    _H, _ = hamiltonian_linop(_geo, hx=HX, hz=HZ, J=1.0)
    E0_EXACT = float(eigsh(_H, k=1, which="SA", return_eigenvectors=False)[0])
    FIELD_FLAGS = f"--hx {HX} --hz {HZ} --exact_E0 {E0_EXACT}"
    POINT_TAG   = f"hx{HX}_hz{HZ}"
    print(f"OBC L={L} on-the-fly ED:  E0_exact = {E0_EXACT:.6f}  (N={_geo.N})")
else:
    # L=2 PBC: use the hardcoded --hz_preset reference (sets both h_z and E_exact).
    FIELD_FLAGS = f"--hx {HX} --hz_preset {HZ_PRESET}"
    POINT_TAG   = HZ_PRESET

# chunk_size for the local-energy/gradient step (caps peak GPU memory). Threaded
# into every run command below as CHUNK_FLAG.
CHUNK_FLAG  = f"--chunk_size {CHUNK}" if CHUNK else ""

WANDB_GROUP = f"colab-{BC.lower()}-{POINT_TAG}-{ARCH_TAG}"
# arch-named folder so each architecture's runs are kept separate:
#   outputs/gridinv/colab-obc-hx0.2_hz0.2/<NAME>.{json,mpack}
OUT_DIR     = f"outputs/{ARCH_TAG}/colab-{BC.lower()}-{POINT_TAG}"

# Reference menu of TRAINING hyperparameters to try (copy a line into a run cell).
# Note: at larger L you typically need a LARGER diag_shift for a stable SR solve.
DT_LRMIN   = [(0.02, 0.002), (0.01, 0.001)]   # (initial lr, final lr)
DIAG_SHIFT = [1e-4, 1e-3, 1e-2]               # SR / QGT regularization

print(f"bc: {BC}   arch: {ARCH}   field flags: {FIELD_FLAGS}")
print(f"chunk: {CHUNK_FLAG or '(none)'}")
print(f"output folder: {OUT_DIR}/   |   "
      + (f"wandb group: {WANDB_GROUP}" if WANDB else "wandb OFF"))
print("training-hyperparameter menu (copy a line into a run cell):")
for (dt, lrm) in DT_LRMIN:
    for ds in DIAG_SHIFT:
        print(f"  --dt {dt} --lr_min {lrm} --diag_shift {ds}")

## Sanity check — one short run

A quick low-cost run to confirm training works end-to-end with the default
`ToricCNN_gridinv` net. The `[train]` line reports `n_params≈3543` for the default
`[4,4]/[4,4]` net (kernel auto = L = 2 at L=2), and you should see the per-step
`delta = ...` stream falling toward ~1e-3 even in this 50-step smoke run.

In [ ]:
# 6. One short smoke run (uses ARCH_FLAGS = the default [4,4]/[4,4] gridinv net).
wb = f"--wandb_group {WANDB_GROUP}" if WANDB else "--no_wandb"
!python -u -m Three_TC.train --L {L} --bc {BC} --model bosonic --arch {ARCH} \
  {FIELD_FLAGS} --n_iter 50 --n_samples 4096 --n_chains 256 \
  --n_sweeps {N_SWEEPS} --qgt dense --dt 0.02 --lr_min 0.002 --diag_shift 1e-4 \
  {CHUNK_FLAG} {ARCH_FLAGS} --out_dir {OUT_DIR} --name cfg_smoke {wb}

## Run configs — one cell per run  (`ToricCNN_gridinv`)

**This is the reliable path:** each cell is a single fresh `!python` process (no
loop). To run a config: **copy a cell below, set `NONINV` / `INV` / `KERNEL`, edit the
training flags (`--dt --lr_min --diag_shift`), and run it.** Each writes
`OUT_DIR/<NAME>.{json,mpack}` under the **arch-named folder**
`outputs/gridinv/colab-<bc>-<point>/` and logs to `WANDB_GROUP`; `--qgt dense`
keeps the SR solve exact and GPU-safe.

**Knobs.** `NONINV` = pre-Wilson geometry-exact edge blocks (uniform channels).
`INV` = post-Wilson **grid `nn.Conv3D`** hidden widths (a width-1 readout is appended).
`KERNEL` = the invariant grid-conv kernel extent; `0` ⇒ auto = `L` (full span, the
topological-coverage choice — at L=2 that is kernel 2). The `NAME` encodes all three,
so configs don't overwrite each other.

> The three cells below are a starter L=2 sweep (default, wider, deeper). Vary
> `--dt`/`--diag_shift` from the configure-cell menu to confirm the ~1e-3→1e-5 `delta`
> reproduces, matching the `ToricCNN_full` runs.

In [ ]:
# === RUN ToricCNN_gridinv (grid nn.Conv3D invariant block; copy per config) ===
NONINV = [4, 4]     # pre-Wilson edge blocks, UNIFORM channels (geometry-exact)
INV    = [4, 4]     # post-Wilson GRID-conv hidden widths; model appends a width-1 readout
KERNEL = 0          # invariant grid-conv kernel; 0 -> auto = L (full span). At L=2 auto=2.

# --- translate to CLI flags + a tag (don't edit) ---------------------------
assert len(set(NONINV)) == 1, f"noninv channels must be uniform, got {NONINV}"
ks_flag = f"--kernel_size {KERNEL}" if KERNEL else ""
ARCH_FLAGS = (f"--noninv_channels {NONINV[0]} --n_noninv {len(NONINV)} "
              f"--inv_hidden {' '.join(map(str, INV))} {ks_flag}").strip()
NAME    = (f"gridinv-{'-'.join(map(str, NONINV))}_{'-'.join(map(str, INV))}"
           + (f"_k{KERNEL}" if KERNEL else "_kL"))
GROUP   = f"colab-{BC.lower()}-{POINT_TAG}-{ARCH_TAG}"
OUT_DIR = f"outputs/{ARCH_TAG}/colab-{BC.lower()}-{POINT_TAG}"
wb      = f"--wandb_group {GROUP}" if WANDB else "--no_wandb"
print(f"arch {NAME}  |  {ARCH_FLAGS}  ->  {OUT_DIR}")

!python -u -m Three_TC.train --L {L} --bc {BC} --model bosonic --arch ToricCNN_gridinv \
  {FIELD_FLAGS} --n_iter 120 --n_samples {N_SAMPLES} \
  --n_chains {N_CHAINS} --n_sweeps {N_SWEEPS} --qgt dense \
  --dt 0.02 --lr_min 0.002 --diag_shift 1e-4 \
  {CHUNK_FLAG} {ARCH_FLAGS} --out_dir {OUT_DIR} --name {NAME} {wb}

In [ ]:
# === RUN ToricCNN_gridinv (grid nn.Conv3D invariant block; copy per config) ===
NONINV = [8, 8]     # pre-Wilson edge blocks, UNIFORM channels (geometry-exact)
INV    = [8, 8]     # post-Wilson GRID-conv hidden widths; model appends a width-1 readout
KERNEL = 0          # invariant grid-conv kernel; 0 -> auto = L (full span). At L=2 auto=2.

# --- translate to CLI flags + a tag (don't edit) ---------------------------
assert len(set(NONINV)) == 1, f"noninv channels must be uniform, got {NONINV}"
ks_flag = f"--kernel_size {KERNEL}" if KERNEL else ""
ARCH_FLAGS = (f"--noninv_channels {NONINV[0]} --n_noninv {len(NONINV)} "
              f"--inv_hidden {' '.join(map(str, INV))} {ks_flag}").strip()
NAME    = (f"gridinv-{'-'.join(map(str, NONINV))}_{'-'.join(map(str, INV))}"
           + (f"_k{KERNEL}" if KERNEL else "_kL"))
GROUP   = f"colab-{BC.lower()}-{POINT_TAG}-{ARCH_TAG}"
OUT_DIR = f"outputs/{ARCH_TAG}/colab-{BC.lower()}-{POINT_TAG}"
wb      = f"--wandb_group {GROUP}" if WANDB else "--no_wandb"
print(f"arch {NAME}  |  {ARCH_FLAGS}  ->  {OUT_DIR}")

!python -u -m Three_TC.train --L {L} --bc {BC} --model bosonic --arch ToricCNN_gridinv \
  {FIELD_FLAGS} --n_iter 120 --n_samples {N_SAMPLES} \
  --n_chains {N_CHAINS} --n_sweeps {N_SWEEPS} --qgt dense \
  --dt 0.02 --lr_min 0.002 --diag_shift 1e-4 \
  {CHUNK_FLAG} {ARCH_FLAGS} --out_dir {OUT_DIR} --name {NAME} {wb}

In [ ]:
# === RUN ToricCNN_gridinv (grid nn.Conv3D invariant block; copy per config) ===
NONINV = [4, 4]     # pre-Wilson edge blocks, UNIFORM channels (geometry-exact)
INV    = [8, 8, 8]  # post-Wilson GRID-conv hidden widths; model appends a width-1 readout
KERNEL = 0          # invariant grid-conv kernel; 0 -> auto = L (full span). At L=2 auto=2.

# --- translate to CLI flags + a tag (don't edit) ---------------------------
assert len(set(NONINV)) == 1, f"noninv channels must be uniform, got {NONINV}"
ks_flag = f"--kernel_size {KERNEL}" if KERNEL else ""
ARCH_FLAGS = (f"--noninv_channels {NONINV[0]} --n_noninv {len(NONINV)} "
              f"--inv_hidden {' '.join(map(str, INV))} {ks_flag}").strip()
NAME    = (f"gridinv-{'-'.join(map(str, NONINV))}_{'-'.join(map(str, INV))}"
           + (f"_k{KERNEL}" if KERNEL else "_kL"))
GROUP   = f"colab-{BC.lower()}-{POINT_TAG}-{ARCH_TAG}"
OUT_DIR = f"outputs/{ARCH_TAG}/colab-{BC.lower()}-{POINT_TAG}"
wb      = f"--wandb_group {GROUP}" if WANDB else "--no_wandb"
print(f"arch {NAME}  |  {ARCH_FLAGS}  ->  {OUT_DIR}")

!python -u -m Three_TC.train --L {L} --bc {BC} --model bosonic --arch ToricCNN_gridinv \
  {FIELD_FLAGS} --n_iter 120 --n_samples {N_SAMPLES} \
  --n_chains {N_CHAINS} --n_sweeps {N_SWEEPS} --qgt dense \
  --dt 0.02 --lr_min 0.002 --diag_shift 1e-4 \
  {CHUNK_FLAG} {ARCH_FLAGS} --out_dir {OUT_DIR} --name {NAME} {wb}

## CNN benchmark — geometry-exact convs, NO Wilson invariance

`ToricCNN_full`'s ~1e-5 errors need a baseline. `GeoCNN` uses the **same**
`KernelManager3D`/`GeoConv3D` kernel and the same depth, but **drops the Wilson
4-product** — so it is translation-equivariant but *not* `A_v`-invariant. A/B-ing
it against `ToricCNN_full` at matched params/depth isolates how much the *vertex
invariance* (not the geometry-exact gather) is worth. Works for OBC and PBC.

Run the param-count helper first to pick `CNN_HIDDEN` close to your full-net size,
then run the benchmark cell (copy it per config, change `CNN_HIDDEN`).

In [ ]:
# === Param-count helper: match nets at comparable size ====================
from Three_TC.builders import build_state

def _nparams(arch, **kw):
    cfg = {"L": L, "bc": BC, "hx": HX, "hz": HZ, "arch": arch,
           "n_samples": 256, "n_chains": 16, **kw}
    *_, vs, _ = build_state(cfg)
    return int(vs.n_parameters)

# default gridinv net (this notebook's arch) and a couple of bigger variants:
print("gridinv [4,4]/[4,4] kL:",
      _nparams("ToricCNN_gridinv", noninv_channels=4, n_noninv=2, inv_hidden=(4, 4)))
print("gridinv [8,8]/[8,8] kL:",
      _nparams("ToricCNN_gridinv", noninv_channels=8, n_noninv=2, inv_hidden=(8, 8)))
# geometry-exact references to A/B against (same Wilson, different invariant block):
print("ToricCNN_full [8,8]/[4,4,4]:",
      _nparams("ToricCNN_full", noninv_channels=8, n_noninv=2, inv_hidden=(4, 4)))
for h in [(8, 8, 8), (12, 12)]:
    print(f"GeoCNN {h} (no Wilson):", _nparams("GeoCNN", cnn_hidden=h))

In [ ]:
# === RUN the CNN benchmark — geometry-exact convs, NO Wilson (copy per cfg) ==
CNN_HIDDEN = [8, 8, 8]   # edge-conv channel widths; depth = len+1 (width-1 readout)

# --- flags + tag (don't edit) ---------------------------------------------
cnn_flags = f"--cnn_hidden {' '.join(map(str, CNN_HIDDEN))}"
NAME    = f"geocnn-{'-'.join(map(str, CNN_HIDDEN))}"
GROUP   = f"colab-{BC.lower()}-{POINT_TAG}"
OUT_DIR = f"outputs/{GROUP}"
wb      = f"--wandb_group {GROUP}" if WANDB else "--no_wandb"
print(f"arch {NAME}  |  {cnn_flags}  ->  {OUT_DIR}")

!python -u -m Three_TC.train --L {L} --bc {BC} --model bosonic --arch GeoCNN \
  {FIELD_FLAGS} --n_iter {N_ITER} --n_samples {N_SAMPLES} \
  --n_chains {N_CHAINS} --n_sweeps {N_SWEEPS} --qgt dense \
  --dt 0.02 --lr_min 0.002 --diag_shift 1e-4 \
  {CHUNK_FLAG} {cnn_flags} --out_dir {OUT_DIR} --name {NAME} {wb}

## Notes

**Default arch.** `ToricCNN_gridinv` — Wilson 4-product → fold flux onto the cube-cell
grid → standard `nn.Conv3D` invariant block (kernel auto = `L`). Vary the net per run
via `NONINV` / `INV` / `KERNEL` in the run cells; vary training via `--dt --lr_min
--diag_shift`. To compare a *different* architecture, set `ARCH`/`ARCH_TAG` in the
configure cell — outputs land in `outputs/<ARCH_TAG>/…`, so architectures never mix.

**Why one cell per run:** each `!python` call is an isolated fresh process, which is
what runs reliably on Colab; the automated loop could skip runs, so it's unrolled.

**Outputs.** Each run writes `OUT_DIR/<NAME>.json` (config + curve + final
observables) and `OUT_DIR/<NAME>.mpack` (weights) into the **arch-named folder**
`outputs/gridinv/colab-<bc>-<point>/`, so a sweep accumulates side by side and
different architectures stay in separate trees. The Colab VM is ephemeral — download
keepers with the last cell, or rely on W&B.

**Boundary conditions.** `BC = "OBC"` switches every run to open boundaries via `--bc OBC` and diagonalizes the L=2 OBC ground state on the fly for the `delta` FOM (no preset needed); PBC keeps using `--hz_preset`. `BC` is folded into `WANDB_GROUP`/`OUT_DIR`, so OBC and PBC runs never share a folder. `gridinv` supports both BC (PBC uses CIRCULAR grid-conv padding, OBC uses zero padding + a masked readout).

**CNN benchmark.** `GeoCNN` (the cells above) is the no-Wilson control: same `GeoConv3D` kernel and depth but without the `A_v`-invariant flux product. Match it to your full net with the param-count helper, then compare `delta` — a clear gap is the evidence that the Wilson invariance, not the kernel, drives the ~1e-5 errors.

In [ ]:
# Optional: zip + download THIS run folder before the VM recycles.
from google.colab import files
!zip -qr /content/{WANDB_GROUP}.zip {OUT_DIR} && echo "zipped {OUT_DIR}"
files.download(f"/content/{WANDB_GROUP}.zip")